# Build SEC Semiconductor Numerical + Filing-Text Data

This notebook downloads official SEC data for:

- Broadcom (`AVGO`)
- NVIDIA (`NVDA`)
- Advanced Micro Devices (`AMD`)
- Intel (`INTC`)
- Micron (`MU`)

It produces:

| Output | Purpose |
|---|---|
| `sec_filings_metadata.parquet` | Filing dates, forms, periods, accession numbers, and SEC URLs |
| `sec_numeric_facts_long.parquet` | Auditable raw XBRL facts selected from SEC Company Facts |
| `sec_filing_text.parquet` | Extracted MD&A, Risk Factors, Business, and combined filing text |
| `semiconductor_sec_numeric_text.parquet` | One model-ready company-quarter row combining numeric and written data |
| CSV versions | Easy inspection and compatibility |
| `data_quality_report.csv` | Coverage and validation checks |

The model-ready file uses **quarterized flow values**. For example, when a 10-K reports annual revenue or a 10-Q reports year-to-date operating cash flow, the notebook subtracts the preceding year-to-date value when possible to estimate the individual quarter.

## Important

Before running, replace `SEC_USER_AGENT` with your real name and email. The SEC requires automated clients to identify themselves.

Run this notebook in Google Colab from top to bottom.

In [ ]:
# Colab dependencies
!pip -q install pandas numpy pyarrow requests beautifulsoup4 lxml tqdm

## 1. Configuration

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
import hashlib
import html as html_lib
import json
import re
import time

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# REQUIRED: replace with your real identity before using SEC EDGAR.
SEC_USER_AGENT = "YOUR NAME your.email@example.com"

# Recent history to collect. Twelve filings normally gives about three years
# per company, although fiscal calendars and amended filings can vary.
FILINGS_PER_COMPANY = 16
FORMS = {"10-Q", "10-K"}

# Narrative text controls.
MAX_SECTION_CHARACTERS = 150_000
SAVE_CLEAN_FULL_TEXT = False

# SEC fair-access guardrails.
MINIMUM_SECONDS_BETWEEN_REQUESTS = 0.20
REQUEST_TIMEOUT_SECONDS = 90
MAX_RETRIES = 5

OUTPUT_DIR = Path("/content/sec_semiconductor_numeric_text")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

COMPANIES = pd.DataFrame(
    [
        {
            "company_name": "Broadcom Inc.",
            "ticker": "AVGO",
            "cik": "1730168",
        },
        {
            "company_name": "NVIDIA Corporation",
            "ticker": "NVDA",
            "cik": "1045810",
        },
        {
            "company_name": "Advanced Micro Devices, Inc.",
            "ticker": "AMD",
            "cik": "2488",
        },
        {
            "company_name": "Intel Corporation",
            "ticker": "INTC",
            "cik": "50863",
        },
        {
            "company_name": "Micron Technology, Inc.",
            "ticker": "MU",
            "cik": "723125",
        },
    ]
)

display(COMPANIES)

## 2. SEC client with caching, identification, retries, and rate limiting

The cache lets you restart the notebook without repeatedly downloading the same files.

In [ ]:
def valid_user_agent(value: str) -> bool:
    lowered = value.lower()
    return (
        "your name" not in lowered
        and "example.com" not in lowered
        and "@" in value
        and len(value.strip()) >= 8
    )


if not valid_user_agent(SEC_USER_AGENT):
    raise ValueError(
        "Replace SEC_USER_AGENT with your real name and email address."
    )


@dataclass
class SecClient:
    user_agent: str
    cache_dir: Path
    minimum_delay: float = MINIMUM_SECONDS_BETWEEN_REQUESTS

    def __post_init__(self) -> None:
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": self.user_agent,
                "Accept-Encoding": "gzip, deflate",
                "Host": "data.sec.gov",
            }
        )
        self.last_request_time = 0.0
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _cache_path(self, url: str, suffix: str) -> Path:
        digest = hashlib.sha256(url.encode("utf-8")).hexdigest()
        return self.cache_dir / f"{digest}{suffix}"

    def _wait(self) -> None:
        elapsed = time.monotonic() - self.last_request_time
        remaining = self.minimum_delay - elapsed
        if remaining > 0:
            time.sleep(remaining)

    def get_bytes(self, url: str, suffix: str) -> bytes:
        cache_path = self._cache_path(url, suffix)
        if cache_path.exists():
            return cache_path.read_bytes()

        last_error: Exception | None = None
        for attempt in range(MAX_RETRIES):
            self._wait()
            try:
                response = self.session.get(
                    url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )
                self.last_request_time = time.monotonic()

                if response.status_code == 429:
                    time.sleep(min(60, 2 ** (attempt + 2)))
                    continue

                response.raise_for_status()
                cache_path.write_bytes(response.content)
                return response.content

            except Exception as exc:
                last_error = exc
                time.sleep(min(30, 2 ** attempt))

        raise RuntimeError(f"SEC request failed: {url}") from last_error

    def get_json(self, url: str) -> dict[str, Any]:
        return json.loads(
            self.get_bytes(url, ".json").decode("utf-8")
        )

    def get_html(self, url: str) -> str:
        return self.get_bytes(url, ".html").decode(
            "utf-8", errors="ignore"
        )


sec = SecClient(
    user_agent=SEC_USER_AGENT,
    cache_dir=CACHE_DIR,
)

## 3. Download filing histories

The SEC Submissions API provides form type, filing date, reporting period, accession number, and primary document name.

In [ ]:
def recent_filings_from_payload(payload: dict[str, Any]) -> pd.DataFrame:
    recent = payload.get("filings", {}).get("recent", {})
    if not recent:
        return pd.DataFrame()

    lengths = [
        len(value)
        for value in recent.values()
        if isinstance(value, list)
    ]
    if not lengths:
        return pd.DataFrame()

    row_count = max(lengths)
    rows = []
    for index in range(row_count):
        row = {}
        for key, values in recent.items():
            row[key] = (
                values[index]
                if isinstance(values, list) and index < len(values)
                else None
            )
        rows.append(row)
    return pd.DataFrame(rows)


def older_filings_from_payload(
    payload: dict[str, Any],
) -> list[pd.DataFrame]:
    frames = []
    for file_info in payload.get("filings", {}).get("files", []):
        name = file_info.get("name")
        if not name:
            continue

        older = sec.get_json(
            f"https://data.sec.gov/submissions/{name}"
        )
        # Older files store the arrays at the root.
        if older and "accessionNumber" in older:
            row_count = len(older["accessionNumber"])
            rows = []
            for index in range(row_count):
                row = {}
                for key, values in older.items():
                    row[key] = (
                        values[index]
                        if isinstance(values, list)
                        and index < len(values)
                        else None
                    )
                rows.append(row)
            frames.append(pd.DataFrame(rows))
    return frames


filing_frames = []

for company in tqdm(
    COMPANIES.itertuples(index=False),
    total=len(COMPANIES),
    desc="Downloading filing histories",
):
    cik_padded = str(company.cik).zfill(10)
    payload = sec.get_json(
        f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    )

    frames = [recent_filings_from_payload(payload)]
    frames.extend(older_filings_from_payload(payload))
    company_filings = pd.concat(
        [frame for frame in frames if not frame.empty],
        ignore_index=True,
    )

    company_filings["company_name"] = company.company_name
    company_filings["ticker"] = company.ticker
    company_filings["cik"] = str(int(company.cik))
    filing_frames.append(company_filings)

all_filings = pd.concat(filing_frames, ignore_index=True)

all_filings["filing_date"] = pd.to_datetime(
    all_filings["filingDate"], errors="coerce"
)
all_filings["quarter_end"] = pd.to_datetime(
    all_filings["reportDate"], errors="coerce"
)
all_filings["accession_number"] = all_filings[
    "accessionNumber"
].astype(str)
all_filings["form"] = all_filings["form"].astype(str)
all_filings["primary_document"] = all_filings[
    "primaryDocument"
].astype(str)

filings = (
    all_filings[
        all_filings["form"].isin(FORMS)
        & all_filings["quarter_end"].notna()
        & all_filings["filing_date"].notna()
    ]
    .sort_values(
        ["ticker", "filing_date", "accession_number"],
        ascending=[True, False, False],
    )
    .drop_duplicates(
        ["ticker", "form", "quarter_end"],
        keep="first",
    )
    .groupby("ticker", group_keys=False)
    .head(FILINGS_PER_COMPANY)
    .sort_values(["ticker", "quarter_end"])
    .reset_index(drop=True)
)

def filing_archive_url(row: pd.Series) -> str:
    cik_number = str(int(row["cik"]))
    accession_compact = row["accession_number"].replace("-", "")
    return (
        "https://www.sec.gov/Archives/edgar/data/"
        f"{cik_number}/{accession_compact}/"
        f"{row['primary_document']}"
    )

filings["filing_url"] = filings.apply(
    filing_archive_url, axis=1
)

metadata_columns = [
    "company_name",
    "ticker",
    "cik",
    "form",
    "filing_date",
    "quarter_end",
    "accession_number",
    "primary_document",
    "filing_url",
    "fiscalYearEnd",
    "isInlineXBRL",
    "isXBRL",
]
metadata_columns = [
    column for column in metadata_columns
    if column in filings.columns
]
filings = filings[metadata_columns]

display(
    filings.groupby("ticker")
    .agg(
        filings=("accession_number", "size"),
        earliest_period=("quarter_end", "min"),
        latest_period=("quarter_end", "max"),
    )
    .reset_index()
)

## 4. Download and flatten official XBRL facts

The notebook preserves a long table of raw facts so every selected value can be audited back to its SEC tag, unit, period, form, filing date, and accession number.

In [ ]:
# Canonical model fields and common US-GAAP tag alternatives.
# The first available valid tag is not blindly selected; all matching facts
# are retained and the filing-level selection logic chooses the best context.
CONCEPT_ALIASES = {
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet",
    ],
    "gross_profit": [
        "GrossProfit",
    ],
    "operating_income": [
        "OperatingIncomeLoss",
    ],
    "net_income": [
        "NetIncomeLoss",
        "ProfitLoss",
    ],
    "research_and_development": [
        "ResearchAndDevelopmentExpense",
    ],
    "stock_based_compensation": [
        "ShareBasedCompensation",
        "ShareBasedCompensationArrangementByShareBasedPaymentAwardEquityInstrumentsOtherThanOptionsGrantsInPeriodTotal",
    ],
    "operating_cash_flow": [
        "NetCashProvidedByUsedInOperatingActivities",
    ],
    "capital_expenditures": [
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "PaymentsForAdditionsToPropertyPlantAndEquipment",
    ],
    "assets": [
        "Assets",
    ],
    "current_assets": [
        "AssetsCurrent",
    ],
    "cash_and_equivalents": [
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
    ],
    "inventory": [
        "InventoryNet",
    ],
    "accounts_receivable": [
        "AccountsReceivableNetCurrent",
        "AccountsNotesAndLoansReceivableNetCurrent",
    ],
    "property_plant_equipment": [
        "PropertyPlantAndEquipmentNet",
    ],
    "current_liabilities": [
        "LiabilitiesCurrent",
    ],
    "liabilities": [
        "Liabilities",
    ],
    "deferred_revenue_current": [
        "ContractWithCustomerLiabilityCurrent",
        "DeferredRevenueCurrent",
    ],
}

TAG_TO_CANONICAL = {
    tag: canonical
    for canonical, tags in CONCEPT_ALIASES.items()
    for tag in tags
}

DURATION_CONCEPTS = {
    "revenue",
    "gross_profit",
    "operating_income",
    "net_income",
    "research_and_development",
    "stock_based_compensation",
    "operating_cash_flow",
    "capital_expenditures",
}

INSTANT_CONCEPTS = set(CONCEPT_ALIASES) - DURATION_CONCEPTS

facts_frames = []

for company in tqdm(
    COMPANIES.itertuples(index=False),
    total=len(COMPANIES),
    desc="Downloading Company Facts",
):
    cik_padded = str(company.cik).zfill(10)
    payload = sec.get_json(
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik_padded}.json"
    )

    us_gaap = payload.get("facts", {}).get("us-gaap", {})

    rows = []
    for tag, fact_payload in us_gaap.items():
        canonical = TAG_TO_CANONICAL.get(tag)
        if canonical is None:
            continue

        label = fact_payload.get("label")
        description = fact_payload.get("description")

        for unit, observations in fact_payload.get(
            "units", {}
        ).items():
            if unit not in {"USD", "shares", "pure"}:
                continue

            for observation in observations:
                row = {
                    "company_name": company.company_name,
                    "ticker": company.ticker,
                    "cik": str(int(company.cik)),
                    "canonical_concept": canonical,
                    "tag": tag,
                    "label": label,
                    "description": description,
                    "unit": unit,
                }
                row.update(observation)
                rows.append(row)

    facts_frames.append(pd.DataFrame(rows))

facts_long = pd.concat(facts_frames, ignore_index=True)

for column in ["start", "end", "filed"]:
    if column in facts_long.columns:
        facts_long[column] = pd.to_datetime(
            facts_long[column], errors="coerce"
        )

facts_long["accession_number"] = facts_long["accn"].astype(
    str
)
facts_long["duration_days"] = (
    facts_long["end"] - facts_long["start"]
).dt.days

selected_accessions = set(filings["accession_number"])
facts_long = facts_long[
    facts_long["accession_number"].isin(selected_accessions)
].copy()

facts_long = facts_long.sort_values(
    [
        "ticker",
        "accession_number",
        "canonical_concept",
        "end",
        "duration_days",
        "filed",
    ]
).reset_index(drop=True)

print("Selected raw XBRL observations:", len(facts_long))
display(
    facts_long[
        [
            "ticker",
            "accession_number",
            "canonical_concept",
            "tag",
            "start",
            "end",
            "duration_days",
            "val",
            "unit",
            "form",
            "fy",
            "fp",
        ]
    ].head(20)
)

## 5. Select and quarterize numeric values

Selection rules:

- Balance-sheet concepts use the instant value closest to the filing period end.
- Duration concepts prefer a true single-quarter value.
- When only year-to-date or annual values are available, the notebook subtracts the preceding cumulative value with the same start date.
- Every field retains its source tag, reported duration, and derivation method.

In [ ]:
def period_distance_days(
    dates: pd.Series,
    target: pd.Timestamp,
) -> pd.Series:
    return (dates - target).abs().dt.days


def select_instant_fact(
    concept_facts: pd.DataFrame,
    period_end: pd.Timestamp,
) -> pd.Series | None:
    candidates = concept_facts[
        concept_facts["end"].notna()
    ].copy()
    if candidates.empty:
        return None

    candidates["end_distance"] = period_distance_days(
        candidates["end"], period_end
    )
    candidates = candidates.sort_values(
        ["end_distance", "filed", "tag"],
        ascending=[True, False, True],
    )
    return candidates.iloc[0]


def select_duration_fact(
    concept_facts: pd.DataFrame,
    period_end: pd.Timestamp,
    form: str,
) -> pd.Series | None:
    candidates = concept_facts[
        concept_facts["start"].notna()
        & concept_facts["end"].notna()
        & concept_facts["duration_days"].notna()
    ].copy()
    if candidates.empty:
        return None

    # Facts ending at the period end are safest. Permit a small calendar
    # tolerance because fiscal-period dates can differ slightly.
    candidates["end_distance"] = period_distance_days(
        candidates["end"], period_end
    )
    candidates = candidates[
        candidates["end_distance"] <= 10
    ].copy()
    if candidates.empty:
        return None

    if form == "10-Q":
        # Prefer an explicitly reported single quarter. If unavailable,
        # retain the shortest cumulative period and quarterize later.
        candidates["quarter_distance"] = (
            candidates["duration_days"] - 91
        ).abs()
        candidates = candidates.sort_values(
            [
                "quarter_distance",
                "duration_days",
                "filed",
                "tag",
            ],
            ascending=[True, True, False, True],
        )
    else:
        # Annual filings usually contain both annual and comparative facts.
        candidates["annual_distance"] = (
            candidates["duration_days"] - 365
        ).abs()
        candidates = candidates.sort_values(
            [
                "annual_distance",
                "duration_days",
                "filed",
                "tag",
            ],
            ascending=[True, False, False, True],
        )

    return candidates.iloc[0]


def find_preceding_cumulative_fact(
    current: pd.Series,
    all_concept_facts: pd.DataFrame,
) -> pd.Series | None:
    if pd.isna(current.get("start")) or pd.isna(
        current.get("end")
    ):
        return None

    candidates = all_concept_facts[
        (all_concept_facts["ticker"] == current["ticker"])
        & (
            all_concept_facts["canonical_concept"]
            == current["canonical_concept"]
        )
        & (all_concept_facts["start"] == current["start"])
        & (all_concept_facts["end"] < current["end"])
        & (
            all_concept_facts["filed"]
            <= current["filed"]
        )
        & all_concept_facts["duration_days"].notna()
    ].copy()

    # The preceding cumulative period should be approximately one fiscal
    # quarter shorter than the current cumulative period.
    candidates = candidates[
        candidates["duration_days"]
        < current["duration_days"] - 45
    ]
    if candidates.empty:
        return None

    candidates["duration_gap"] = (
        current["duration_days"]
        - candidates["duration_days"]
    ).abs()
    candidates = candidates[
        candidates["duration_gap"].between(60, 125)
    ]
    if candidates.empty:
        return None

    return candidates.sort_values(
        ["end", "filed"],
        ascending=[False, False],
    ).iloc[0]


numeric_records = []

for filing in tqdm(
    filings.itertuples(index=False),
    total=len(filings),
    desc="Selecting numeric values",
):
    filing_facts = facts_long[
        facts_long["accession_number"]
        == filing.accession_number
    ]

    record: dict[str, Any] = {
        "company_name": filing.company_name,
        "ticker": filing.ticker,
        "cik": filing.cik,
        "form": filing.form,
        "filing_date": filing.filing_date,
        "quarter_end": filing.quarter_end,
        "accession_number": filing.accession_number,
        "primary_document": filing.primary_document,
        "filing_url": filing.filing_url,
    }

    for concept in CONCEPT_ALIASES:
        concept_facts = filing_facts[
            filing_facts["canonical_concept"] == concept
        ]

        if concept in INSTANT_CONCEPTS:
            selected = select_instant_fact(
                concept_facts,
                filing.quarter_end,
            )
        else:
            selected = select_duration_fact(
                concept_facts,
                filing.quarter_end,
                filing.form,
            )

        if selected is None:
            record[concept] = np.nan
            record[f"{concept}_reported"] = np.nan
            record[f"{concept}_source_tag"] = None
            record[f"{concept}_duration_days"] = np.nan
            record[f"{concept}_derivation"] = "missing"
            continue

        reported_value = float(selected["val"])
        final_value = reported_value
        derivation = "reported_instant"

        if concept in DURATION_CONCEPTS:
            duration = float(selected["duration_days"])
            if duration <= 125:
                derivation = "reported_single_quarter"
            else:
                previous = find_preceding_cumulative_fact(
                    selected,
                    facts_long,
                )
                if previous is not None:
                    final_value = (
                        reported_value - float(previous["val"])
                    )
                    derivation = (
                        "derived_current_cumulative_minus_"
                        "preceding_cumulative"
                    )
                else:
                    # Keep the reported value for auditability but do not
                    # mislabel an annual/YTD value as a quarterly feature.
                    final_value = np.nan
                    derivation = (
                        "cumulative_value_not_quarterized"
                    )

        record[concept] = final_value
        record[f"{concept}_reported"] = reported_value
        record[f"{concept}_source_tag"] = selected["tag"]
        record[f"{concept}_duration_days"] = selected[
            "duration_days"
        ]
        record[f"{concept}_derivation"] = derivation

    numeric_records.append(record)

numeric_wide = pd.DataFrame(numeric_records).sort_values(
    ["ticker", "quarter_end"]
).reset_index(drop=True)

# Model-oriented financial features.
numeric_wide["gross_margin"] = (
    numeric_wide["gross_profit"]
    / numeric_wide["revenue"]
)
numeric_wide["operating_margin"] = (
    numeric_wide["operating_income"]
    / numeric_wide["revenue"]
)
numeric_wide["cash_flow_margin"] = (
    numeric_wide["operating_cash_flow"]
    / numeric_wide["revenue"]
)
numeric_wide["inventory_to_revenue"] = (
    numeric_wide["inventory"]
    / numeric_wide["revenue"]
)
numeric_wide["receivables_to_revenue"] = (
    numeric_wide["accounts_receivable"]
    / numeric_wide["revenue"]
)
numeric_wide["capex_to_revenue"] = (
    numeric_wide["capital_expenditures"]
    / numeric_wide["revenue"]
)
numeric_wide["research_and_development_ratio"] = (
    numeric_wide["research_and_development"]
    / numeric_wide["revenue"]
)

grouped_numeric = numeric_wide.groupby(
    "ticker", group_keys=False
)
numeric_wide["revenue_yoy_growth"] = (
    numeric_wide["revenue"]
    / grouped_numeric["revenue"].shift(4)
    - 1.0
)
numeric_wide["previous_yoy_growth"] = grouped_numeric[
    "revenue_yoy_growth"
].shift(1)
numeric_wide["revenue_momentum"] = (
    numeric_wide["revenue_yoy_growth"]
    - numeric_wide["previous_yoy_growth"]
)
numeric_wide["next_quarter_growth"] = grouped_numeric[
    "revenue_yoy_growth"
].shift(-1)
numeric_wide["future_growth_change"] = (
    numeric_wide["next_quarter_growth"]
    - numeric_wide["revenue_yoy_growth"]
)
numeric_wide["feature_cutoff_date"] = numeric_wide[
    "filing_date"
]
numeric_wide["label_available_date"] = grouped_numeric[
    "filing_date"
].shift(-1)

display(
    numeric_wide[
        [
            "ticker",
            "form",
            "quarter_end",
            "filing_date",
            "revenue",
            "gross_profit",
            "gross_margin",
            "inventory",
            "accounts_receivable",
            "operating_cash_flow",
            "revenue_yoy_growth",
            "revenue_momentum",
        ]
    ].tail(20)
)

## 6. Download and extract filing writing

The output includes:

- `mda_text`: Management's Discussion and Analysis
- `risk_factors_text`: Risk Factors
- `business_text`: Business section, mainly useful for 10-K filings
- `sec_text`: combined model input
- optionally `clean_full_text`

Section extraction is heuristic because companies can format filings differently. The source URL and full filing metadata remain in every row for verification.

In [ ]:
def clean_filing_html(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "lxml")

    for tag in soup(
        ["script", "style", "noscript", "svg", "ix:header"]
    ):
        tag.decompose()

    # Narrative modeling usually benefits from removing large financial
    # tables. Numeric values are collected separately from XBRL.
    for table in soup.find_all("table"):
        table.decompose()

    text = soup.get_text(" ")
    text = html_lib.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def section_candidates(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
    minimum_characters: int = 500,
) -> list[str]:
    starts = []
    for pattern in start_patterns:
        starts.extend(re.finditer(pattern, text, flags=re.I))

    ends = []
    for pattern in end_patterns:
        ends.extend(re.finditer(pattern, text, flags=re.I))

    candidates = []
    for start_match in starts:
        possible_ends = [
            end_match
            for end_match in ends
            if end_match.start()
            > start_match.end() + minimum_characters
        ]
        if not possible_ends:
            continue

        nearest_end = min(
            possible_ends,
            key=lambda match: match.start(),
        )
        candidate = text[
            start_match.start():nearest_end.start()
        ].strip()

        if (
            minimum_characters
            <= len(candidate)
            <= MAX_SECTION_CHARACTERS
        ):
            candidates.append(candidate)

    return candidates


def longest_section(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
) -> str:
    candidates = section_candidates(
        text,
        start_patterns,
        end_patterns,
    )
    return max(candidates, key=len) if candidates else ""


def extract_sections(
    clean_text: str,
    form: str,
) -> dict[str, str]:
    if form == "10-K":
        mda_start = [
            r"\bitem\s+7[\.\:\-\s]+management[’']?s?\s+discussion",
        ]
        mda_end = [
            r"\bitem\s+7a[\.\:\-\s]+",
            r"\bitem\s+8[\.\:\-\s]+financial",
        ]
        business_start = [
            r"\bitem\s+1[\.\:\-\s]+business\b",
        ]
        business_end = [
            r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
            r"\bitem\s+1b[\.\:\-\s]+",
        ]
    else:
        mda_start = [
            r"\bitem\s+2[\.\:\-\s]+management[’']?s?\s+discussion",
        ]
        mda_end = [
            r"\bitem\s+3[\.\:\-\s]+quantitative",
            r"\bitem\s+4[\.\:\-\s]+controls",
        ]
        business_start = []
        business_end = []

    risk_start = [
        r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
    ]
    risk_end = [
        r"\bitem\s+1b[\.\:\-\s]+",
        r"\bitem\s+1c[\.\:\-\s]+",
        r"\bitem\s+2[\.\:\-\s]+",
    ]

    mda = longest_section(
        clean_text, mda_start, mda_end
    )
    risk = longest_section(
        clean_text, risk_start, risk_end
    )
    business = (
        longest_section(
            clean_text,
            business_start,
            business_end,
        )
        if business_start
        else ""
    )

    combined_parts = []
    if business:
        combined_parts.append(
            "BUSINESS SECTION\n" + business
        )
    if mda:
        combined_parts.append(
            "MANAGEMENT DISCUSSION AND ANALYSIS\n" + mda
        )
    if risk:
        combined_parts.append(
            "RISK FACTORS\n" + risk
        )

    # Fallback for filings whose headings are not recognized.
    if not combined_parts:
        combined_parts.append(
            "FILING NARRATIVE\n"
            + clean_text[:MAX_SECTION_CHARACTERS]
        )

    return {
        "business_text": business,
        "mda_text": mda,
        "risk_factors_text": risk,
        "sec_text": "\n\n".join(combined_parts),
    }


text_records = []
text_failures = []

# The archives use www.sec.gov rather than data.sec.gov.
sec.session.headers["Host"] = "www.sec.gov"

for filing in tqdm(
    filings.itertuples(index=False),
    total=len(filings),
    desc="Downloading filing text",
):
    try:
        raw_html = sec.get_html(filing.filing_url)
        clean_text = clean_filing_html(raw_html)
        sections = extract_sections(
            clean_text,
            filing.form,
        )

        record = {
            "company_name": filing.company_name,
            "ticker": filing.ticker,
            "cik": filing.cik,
            "form": filing.form,
            "filing_date": filing.filing_date,
            "quarter_end": filing.quarter_end,
            "accession_number": filing.accession_number,
            "primary_document": filing.primary_document,
            "filing_url": filing.filing_url,
            **sections,
        }
        if SAVE_CLEAN_FULL_TEXT:
            record["clean_full_text"] = clean_text
        text_records.append(record)

    except Exception as exc:
        text_failures.append(
            {
                "ticker": filing.ticker,
                "accession_number": filing.accession_number,
                "filing_url": filing.filing_url,
                "error": str(exc),
            }
        )

filing_text = pd.DataFrame(text_records)

for column in [
    "business_text",
    "mda_text",
    "risk_factors_text",
    "sec_text",
]:
    if column not in filing_text.columns:
        filing_text[column] = ""
    filing_text[f"{column}_characters"] = (
        filing_text[column].fillna("").str.len()
    )

if text_failures:
    pd.DataFrame(text_failures).to_csv(
        OUTPUT_DIR / "text_download_failures.csv",
        index=False,
    )
    print(
        f"Text failures: {len(text_failures)}. "
        "See text_download_failures.csv."
    )

display(
    filing_text[
        [
            "ticker",
            "form",
            "quarter_end",
            "mda_text_characters",
            "risk_factors_text_characters",
            "business_text_characters",
            "sec_text_characters",
        ]
    ].tail(20)
)

## 7. Merge the numerical and written data

`semiconductor_sec_numeric_text.parquet` is the main model input. It uses the same key fields expected by the financial-plus-text modeling notebook.

In [ ]:
model_dataset = numeric_wide.merge(
    filing_text[
        [
            column
            for column in [
                "accession_number",
                "business_text",
                "mda_text",
                "risk_factors_text",
                "sec_text",
                "business_text_characters",
                "mda_text_characters",
                "risk_factors_text_characters",
                "sec_text_characters",
                "clean_full_text",
            ]
            if column in filing_text.columns
        ]
    ],
    on="accession_number",
    how="left",
)

for column in [
    "business_text",
    "mda_text",
    "risk_factors_text",
    "sec_text",
]:
    model_dataset[column] = model_dataset[
        column
    ].fillna("")

model_dataset["has_sec_text"] = (
    model_dataset["sec_text"].str.len() >= 500
)

# Stable observation ID for traceability.
model_dataset["observation_id"] = (
    model_dataset["ticker"]
    + "_"
    + model_dataset["quarter_end"].dt.strftime("%Y-%m-%d")
    + "_"
    + model_dataset["accession_number"]
)

# Put key identifiers, dates, financial inputs, and text first.
preferred_columns = [
    "observation_id",
    "company_name",
    "ticker",
    "cik",
    "form",
    "quarter_end",
    "filing_date",
    "feature_cutoff_date",
    "label_available_date",
    "accession_number",
    "primary_document",
    "filing_url",
    "revenue",
    "gross_profit",
    "gross_margin",
    "operating_income",
    "operating_margin",
    "net_income",
    "inventory",
    "accounts_receivable",
    "operating_cash_flow",
    "cash_flow_margin",
    "capital_expenditures",
    "capex_to_revenue",
    "research_and_development",
    "research_and_development_ratio",
    "assets",
    "liabilities",
    "cash_and_equivalents",
    "deferred_revenue_current",
    "revenue_yoy_growth",
    "previous_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "future_growth_change",
    "business_text",
    "mda_text",
    "risk_factors_text",
    "sec_text",
    "has_sec_text",
]

preferred_columns = [
    column for column in preferred_columns
    if column in model_dataset.columns
]
remaining_columns = [
    column for column in model_dataset.columns
    if column not in preferred_columns
]
model_dataset = model_dataset[
    preferred_columns + remaining_columns
].sort_values(
    ["ticker", "quarter_end"]
).reset_index(drop=True)

display(model_dataset.head())

## 8. Quality checks and exports

The CSV containing full filing text can be large. Parquet is the recommended format for modeling because it is compressed and preserves types.

In [ ]:
quality_rows = []

for ticker, frame in model_dataset.groupby("ticker"):
    quality_rows.append(
        {
            "ticker": ticker,
            "rows": len(frame),
            "first_quarter_end": frame["quarter_end"].min(),
            "last_quarter_end": frame["quarter_end"].max(),
            "revenue_coverage": frame["revenue"].notna().mean(),
            "gross_profit_coverage": frame[
                "gross_profit"
            ].notna().mean(),
            "inventory_coverage": frame[
                "inventory"
            ].notna().mean(),
            "receivables_coverage": frame[
                "accounts_receivable"
            ].notna().mean(),
            "operating_cash_flow_coverage": frame[
                "operating_cash_flow"
            ].notna().mean(),
            "text_coverage": frame[
                "has_sec_text"
            ].mean(),
            "duplicate_observation_ids": frame[
                "observation_id"
            ].duplicated().sum(),
        }
    )

quality_report = pd.DataFrame(quality_rows)
display(quality_report)

# Parquet outputs.
filings.to_parquet(
    OUTPUT_DIR / "sec_filings_metadata.parquet",
    index=False,
)
facts_long.to_parquet(
    OUTPUT_DIR / "sec_numeric_facts_long.parquet",
    index=False,
)
filing_text.to_parquet(
    OUTPUT_DIR / "sec_filing_text.parquet",
    index=False,
)
model_dataset.to_parquet(
    OUTPUT_DIR
    / "semiconductor_sec_numeric_text.parquet",
    index=False,
)

# CSV outputs. The long and text tables can be large.
filings.to_csv(
    OUTPUT_DIR / "sec_filings_metadata.csv",
    index=False,
)
facts_long.to_csv(
    OUTPUT_DIR / "sec_numeric_facts_long.csv",
    index=False,
)
filing_text.to_csv(
    OUTPUT_DIR / "sec_filing_text.csv",
    index=False,
)
model_dataset.to_csv(
    OUTPUT_DIR
    / "semiconductor_sec_numeric_text.csv",
    index=False,
)
quality_report.to_csv(
    OUTPUT_DIR / "data_quality_report.csv",
    index=False,
)

# A smaller numerical-only CSV is convenient for quick inspection.
numeric_wide.to_csv(
    OUTPUT_DIR / "sec_numeric_quarterly.csv",
    index=False,
)

print("Created files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f" - {path.name}: {path.stat().st_size:,} bytes")

## 9. Download the finished files from Colab

The ZIP contains the data outputs but not the request cache.

In [ ]:
import shutil

zip_base = "/content/SEC_Semiconductor_Numeric_and_Text_Data"
zip_file = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=OUTPUT_DIR,
)

print("ZIP created:", zip_file)

try:
    from google.colab import files
    files.download(zip_file)
except ImportError:
    print("Not running in Colab. Download the ZIP from:", zip_file)

## Using the output in the modeling notebook

Use:

```python
DATA_PATH = Path(
    "/content/sec_semiconductor_numeric_text/"
    "semiconductor_sec_numeric_text.parquet"
)
```

The table already contains `sec_text`, financial features, SEC metadata, accession numbers, filing dates, and model timing fields.

For a fair text-value comparison, evaluate:

- financial only
- TF-IDF text only
- sentence embeddings only
- financial + sentence embeddings

on exactly the same rows with nonempty `sec_text`.